# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

In [78]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [79]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [80]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [81]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
Do not include social media links, such as LinkedIn, Twitter/X, Facebook, Instagram or YouTube.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [82]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.
    
    Links (some might be relative URLs):
    
    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [83]:
print(get_links_user_prompt("https://edwarddonner.com"))


    Here is the list of links on the website https://edwarddonner.com -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative URLs):

    #wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-archi

In [84]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content":
            get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [85]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [86]:
def select_relevant_links(url):
    print(f"Selecting relevant links for URL: {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model = MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [87]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for URL: https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page (founder)',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [88]:
def fetch_page_and_all_relevant_links(url):
    page_content = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{page_content}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link['url'])
    return result

In [89]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

Selecting relevant links for URL: https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links
## Landing Page:

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with over a million 

In [90]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [93]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
    """
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [94]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co/")

ReadTimeout: HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=None)

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co/")

BadRequestError: Error code: 400 - {'error': {'message': "Invalid value for 'content': expected a string, got null.", 'type': 'invalid_request_error', 'param': 'messages.[1].content', 'code': None}}

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [99]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response),
        display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for URL: https://edwarddonner.com/ by calling gpt-5-nano
Found 8 relevant links


# Ed Donner – Company Brochure

---

## About Ed Donner & Nebula.io

Ed Donner is a passionate AI technologist, coder, and entrepreneur, co-founder and CTO of **Nebula.io**, an innovative AI startup revolutionizing recruitment. With a rich background as founder and CEO of the AI startup *untapt* (acquired in 2021) and experience as a Managing Director at JPMorgan, Ed brings deep expertise and a pioneering spirit to the field of AI.

Nebula.io harnesses Generative AI and advanced machine learning to transform how recruiters source, understand, engage, and manage talent. Their patented matching model transcends traditional keyword searches, offering far greater speed and accuracy in connecting people with their ideal roles. Best of all, it’s free to try.

---

## Vision & Mission

Nebula.io is motivated by the Japanese philosophy of **Ikigai** — helping individuals discover their purpose and find work where they’ll thrive both personally and professionally. With 77% of workers feeling uninspired or unengaged, Ed’s mission aims to boost human prosperity by aligning people with roles that ignite fulfillment and success.

---

## Products & Innovations

- **AI-powered Talent Matching**: Patented algorithms provide intuitive, highly accurate matching of candidates to roles without relying on keywords.
- **C4 & Outsmart**: Cutting-edge initiatives involving complex AI models, such as arenas where large language models (LLMs) compete in diplomacy and strategy challenges.
- **Comprehensive AI Curriculum**: Ed has designed top-rated Udemy courses on AI engineering, used by over 1 million students across 194 countries, covering topics like AI coding, agentic engineering, and AI-driven production deployment.

---

## Company Culture

Ed combines a love for deep technical exploration with authenticity and humility. His passion for AI extends beyond work — into music production, community engagement, and education. The culture is one of:

- **Continuous Learning**: Encouraging mastery and open exchange around AI breakthroughs.
- **Curiosity & Experimentation**: Emphasizing hands-on development and real-world problem solving.
- **Impact-Driven Work**: Focused on meaningful outcomes that benefit individuals and society.
- **Welcoming & Collaborative Environment**: Ed’s approachable style invites conversation and mentorship.

---

## Careers & Opportunities

Nebula.io seeks forward-thinking engineers, AI researchers, and product innovators eager to:

- Work at the forefront of AI and machine learning.
- Solve challenging real-world problems, especially in recruitment and talent management.
- Join a mission-driven company passionate about improving people’s lives by unlocking human potential.
- Thrive in a dynamic startup environment backed by an experienced founder dedicated to growth and development.

---

## Connect & Learn More

- **Website**: www.edwarddonner.com  
- **Email**: ed [at] edwarddonner [dot] com  
- **Socials**: LinkedIn | Twitter | Facebook  

Explore Ed’s digital avatar and join the AI conversation. Whether you’re a prospective customer looking to transform hiring, an investor seeking the next AI breakthrough, or a talented individual ready to shape the future — Nebula.io offers a unique opportunity to grow, learn, and impact.

---

## Summary

Ed Donner is a visionary AI leader bridging the gap between cutting-edge technology and meaningful human potential. His company, Nebula.io, is redefining recruitment with AI, aiming to create fulfilling careers globally. Backed by innovation, education, and an inspiring mission, Nebula.io invites you to be part of the AI revolution.